# Project 5 — Extended solution: structured, enum-backed application errors

**Standalone notebook.** This version reimplements the baseline without importing the advanced notebook and adds typed application errors, machine-readable context, safe JSON round-trips, a public catalog, explicit exception chaining, and a small configuration-validation example. Standard library only; tested with Python 3.13.

**Compatibility:** Preserve `AppException.Timeout.throw()`, `AppException.NotAnInteger.throw()`, the four original codes 100/101/200/201, and `ValueError` compatibility for validation errors. This version adds *structured fields* to raised exceptions and accepts `context=` and `cause=` as keyword-only arguments. `Timeout` is a custom `ConnectionException`. The introductory example's code 100 contradicts the completed source enum, so timeout uses code 101 here, too.

**Security and scope:** Structured context must be deliberately selected by callers: do not include credentials, tokens or private user information. We validate that it is JSON-compatible and make a deeply immutable snapshot, but this is a small in-process library rather than a distributed observability platform.

## 1. Structured, immutable context and a common exception hierarchy

`ApplicationError` carries `code`, `key`, `message`, and read-only `context`. Nested mappings are recursively protected and lists become tuples; `to_dict()` yields normal JSON-compatible values. The `ValidationException` has both `ApplicationError` and `ValueError` in its inheritance tree to preserve compatibility with the baseline. `throw(..., cause=err)` uses normal Python exception chaining.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum, unique
import json
import math
from types import MappingProxyType
from typing import Any, Mapping, NoReturn


def _freeze_json(value: Any, *, path: str = "context") -> Any:
    """Make a deep immutable copy of strictly JSON-compatible data."""
    if value is None or isinstance(value, (str, bool)) or type(value) is int:
        return value
    if type(value) is float:
        if not math.isfinite(value):
            raise ValueError(f"{path} must contain finite floats")
        return value
    if isinstance(value, dict):
        if not all(isinstance(key, str) for key in value):
            raise TypeError(f"{path} mapping keys must be strings")
        return MappingProxyType({key: _freeze_json(part, path=f"{path}.{key}")
                                 for key, part in value.items()})
    if isinstance(value, (list, tuple)):
        return tuple(_freeze_json(part, path=f"{path}[{idx}]")
                     for idx, part in enumerate(value))
    raise TypeError(f"{path} must contain only JSON-compatible values")


def _thaw_json(value: Any) -> Any:
    """Convert immutable context into detached, JSON-serializable structures."""
    if isinstance(value, Mapping):
        return {key: _thaw_json(part) for key, part in value.items()}
    if isinstance(value, tuple):
        return [_thaw_json(part) for part in value]
    return value


class ApplicationError(Exception):
    """Base class for structured errors originating from the application catalog."""

    def __init__(self, code: int, key: str, message: str,
                 context: Mapping[str, Any] | None = None) -> None:
        if type(code) is not int or code < 0:
            raise ValueError("code must be a non-negative integer")
        if not isinstance(key, str) or not key:
            raise ValueError("key must be a nonempty string")
        if not isinstance(message, str) or not message.strip():
            raise ValueError("message must be a nonempty string")
        if context is None:
            context = {}
        if not isinstance(context, dict) and not isinstance(context, Mapping):
            raise TypeError("context must be a mapping or None")
        # Convert Mapping to dict before freezing; the nested contents are copied.
        self.context = _freeze_json(dict(context))
        self.code = code
        self.key = key
        self.message = message
        super().__init__(f"{code} - {message}")

    def to_dict(self) -> dict[str, Any]:
        """Return a detached representation suitable for JSON APIs or logs."""
        return {"code": self.code, "key": self.key,
                "message": self.message, "context": _thaw_json(self.context)}

    def to_json(self) -> str:
        """Serialize this exception without traceback or chained-cause leakage."""
        return json.dumps(self.to_dict(), ensure_ascii=False, sort_keys=True,
                          allow_nan=False)


class GenericException(ApplicationError):
    """General application failure."""


class ConnectionException(ApplicationError):
    """Connectivity-related application failure."""


class Timeout(ConnectionException):
    """A configured connection timed out."""


class ValidationException(ApplicationError, ValueError):
    """A structured application error that is also catchable as ValueError."""


@dataclass(frozen=True, slots=True)
class ErrorDefinition:
    """Public, immutable metadata for an error catalog entry."""
    name: str
    code: int
    exception_type: type[ApplicationError]
    message: str

## 2. Central catalog and structured raising

The enum remains the single source of truth for stable codes, classes and messages. You can construct an exception with `create(...)`, raise it with `throw(...)`, look up a member with `AppException(code)` or `AppException['Key']`, or use `from_dict` / `from_json` to reconstruct a known catalog error from a trusted, validated payload. Unknown/mismatched keys or codes are rejected rather than silently relabeled.

In [2]:
@unique
class AppException(Enum):
    Generic = (100, GenericException, "Application exception.")
    Timeout = (101, Timeout, "Timeout connecting to resource.")
    NotAnInteger = (200, ValidationException, "Value must be an integer.")
    NotAList = (201, ValidationException, "Value must be a list.")
    MissingField = (202, ValidationException, "Required field is missing.")
    NotAString = (203, ValidationException, "Value must be a string.")
    InvalidRange = (204, ValidationException, "Value is outside the permitted range.")
    UnexpectedField = (205, ValidationException, "Unexpected configuration field.")

    def __new__(cls, code: int, exception_type: type[ApplicationError], message: str):
        if type(code) is not int or code < 0:
            raise ValueError("Catalog codes must be non-negative integers")
        if not isinstance(exception_type, type) or not issubclass(exception_type, ApplicationError):
            raise TypeError("Catalog exception type must derive from ApplicationError")
        if not isinstance(message, str) or not message.strip():
            raise ValueError("Catalog messages must be nonblank strings")
        obj = object.__new__(cls)
        obj._value_ = code
        obj.exception = exception_type
        obj.message = message
        return obj

    @property
    def code(self) -> int:
        return self.value

    def create(self, message: str | None = None, *,
               context: Mapping[str, Any] | None = None) -> ApplicationError:
        """Build the configured error without raising it."""
        if message is None:
            message = self.message
        if not isinstance(message, str):
            raise TypeError("message must be str or None")
        if not message.strip():
            raise ValueError("message must not be blank")
        return self.exception(self.code, self.name, message, context)

    def throw(self, message: str | None = None, *,
              context: Mapping[str, Any] | None = None,
              cause: Exception | None = None) -> NoReturn:
        """Raise a typed catalog error; optionally chain an original cause."""
        if cause is not None and not isinstance(cause, Exception):
            raise TypeError("cause must be an Exception instance or None")
        error = self.create(message, context=context)
        if cause is None:
            raise error
        raise error from cause

    @classmethod
    def catalog(cls) -> tuple[ErrorDefinition, ...]:
        return tuple(ErrorDefinition(e.name, e.code, e.exception, e.message)
                     for e in cls)

    @classmethod
    def catalog_json(cls) -> str:
        """Export public error definitions for documentation or service contracts."""
        records = [{"key": e.name, "code": e.code, "type": e.exception.__name__,
                    "default_message": e.message} for e in cls]
        return json.dumps(records, indent=2, ensure_ascii=False)

    @classmethod
    def from_dict(cls, payload: Mapping[str, Any]) -> ApplicationError:
        """Reconstruct only an existing error with a matching stable key and code."""
        if not isinstance(payload, Mapping):
            raise TypeError("payload must be a mapping")
        if set(payload) != {"code", "key", "message", "context"}:
            raise ValueError("payload must contain exactly code, key, message, context")
        if type(payload["code"]) is not int:
            raise TypeError("code must be an integer (not bool)")
        if not isinstance(payload["key"], str):
            raise TypeError("key must be a string")
        try:
            member = cls(payload["code"])
        except ValueError as exc:
            raise ValueError("Unknown error code") from exc
        if member.name != payload["key"]:
            raise ValueError("Error code and key do not match")
        return member.create(payload["message"], context=payload["context"])

    @classmethod
    def from_json(cls, serialized: str) -> ApplicationError:
        if not isinstance(serialized, str):
            raise TypeError("serialized data must be a JSON string")
        # Reject nonstandard JSON NaN/Infinity at parse time.
        def reject_constant(value: str) -> NoReturn:
            raise ValueError(f"Nonstandard JSON constant: {value}")
        return cls.from_dict(json.loads(serialized, parse_constant=reject_constant))

## 3. Demonstration: metadata, chaining, lookup, round-trip

The message output stays `CODE - text` for compatibility, while structured information is accessible through named attributes. Serialized errors intentionally omit full tracebacks and chained causes. Include only safe, nonsensitive context in exceptions and logs.

In [3]:
try:
    try:
        raise OSError("socket failed")
    except OSError as underlying:
        AppException.Timeout.throw("Database connection timed out.",
                                   context={"service": "inventory", "attempt": 3,
                                            "retry_delays": [1, 2, 4]},
                                   cause=underlying)
except ConnectionException as error:
    print("Caught:", type(error).__name__, str(error))
    print("Machine-readable:", error.to_dict())
    print("Chained cause:", type(error.__cause__).__name__)
    reconstructed = AppException.from_json(error.to_json())
    assert reconstructed.to_dict() == error.to_dict()
    print("Round trip:", reconstructed.to_json())

print("Available catalog keys:", [definition.name for definition in AppException.catalog()])

Caught: Timeout 101 - Database connection timed out.
Machine-readable: {'code': 101, 'key': 'Timeout', 'message': 'Database connection timed out.', 'context': {'service': 'inventory', 'attempt': 3, 'retry_delays': [1, 2, 4]}}
Chained cause: OSError
Round trip: {"code": 101, "context": {"attempt": 3, "retry_delays": [1, 2, 4], "service": "inventory"}, "key": "Timeout", "message": "Database connection timed out."}
Available catalog keys: ['Generic', 'Timeout', 'NotAnInteger', 'NotAList', 'MissingField', 'NotAString', 'InvalidRange', 'UnexpectedField']


## 4. Applied example: validate a configuration object

A practical caller can use stable error codes to signal incorrect fields and attach structured context. The pure function below returns a normalized result without mutating the input. It validates an expected `retries` integer (0–10), a nonempty `hosts` list of nonblank strings, and an optional positive `timeout_seconds` integer (default 30); unknown keys are rejected. It deliberately **does not** put hostnames or secret values in the error context.

In [4]:
def validate_configuration(config: Mapping[str, Any]) -> dict[str, Any]:
    """Validate, normalize, and detach a small, example service configuration."""
    if not isinstance(config, Mapping):
        AppException.NotAList.throw("Configuration must be a mapping.",
                                    context={"field": "config"})

    allowed = {"retries", "hosts", "timeout_seconds"}
    for key in ("retries", "hosts"):
        if key not in config:
            AppException.MissingField.throw(context={"field": key})
    for key in config:
        if key not in allowed:
            AppException.UnexpectedField.throw(context={"field": str(key)})

    retries = config["retries"]
    if type(retries) is not int:
        AppException.NotAnInteger.throw(context={"field": "retries"})
    if not 0 <= retries <= 10:
        AppException.InvalidRange.throw(context={"field": "retries", "min": 0, "max": 10})

    hosts = config["hosts"]
    if not isinstance(hosts, list):
        AppException.NotAList.throw(context={"field": "hosts"})
    if not hosts:
        AppException.InvalidRange.throw("hosts must not be empty.",
                                        context={"field": "hosts", "min_length": 1})
    for index, host in enumerate(hosts):
        if not isinstance(host, str):
            AppException.NotAString.throw(context={"field": "hosts", "index": index})
        if not host.strip():
            AppException.InvalidRange.throw("Hostnames cannot be blank.",
                                            context={"field": "hosts", "index": index})

    timeout = config.get("timeout_seconds", 30)
    if type(timeout) is not int:
        AppException.NotAnInteger.throw(context={"field": "timeout_seconds"})
    if timeout <= 0:
        AppException.InvalidRange.throw(context={"field": "timeout_seconds", "min": 1})
    return {"retries": retries, "hosts": list(hosts), "timeout_seconds": timeout}


example = validate_configuration({"retries": 3, "hosts": ["node-a", "node-b"]})
print("Normalized config:", example)
try:
    validate_configuration({"retries": True, "hosts": ["node-a"]})
except ValidationException as exc:
    print("Config rejected:", exc.to_dict())

Normalized config: {'retries': 3, 'hosts': ['node-a', 'node-b'], 'timeout_seconds': 30}
Config rejected: {'code': 200, 'key': 'NotAnInteger', 'message': 'Value must be an integer.', 'context': {'field': 'retries'}}


## 5. Tests: catalog, structured errors, JSON, validation and security boundaries

Tests exercise exact legacy codes and catch types, deep copying/immutability, unknown identifiers, corruption/mismatch rejection, chaining, non-JSON context, nonfinite floats, and the applied configuration validator.

In [5]:
import unittest


class TestExtendedExceptionCatalog(unittest.TestCase):
    def test_legacy_codes(self):
        self.assertEqual([(x.name, x.code) for x in list(AppException)[:4]],
                         [("Generic", 100), ("Timeout", 101),
                          ("NotAnInteger", 200), ("NotAList", 201)])

    def test_additional_codes_are_unique(self):
        self.assertEqual(len({x.code for x in AppException}), len(AppException))

    def test_legacy_lookup_by_name(self):
        self.assertIs(AppException["Timeout"], AppException.Timeout)

    def test_legacy_lookup_by_code(self):
        self.assertIs(AppException(200), AppException.NotAnInteger)

    def test_timeout_caught_as_connection_exception(self):
        with self.assertRaises(ConnectionException):
            AppException.Timeout.throw()

    def test_timeout_caught_as_application_error(self):
        with self.assertRaises(ApplicationError):
            AppException.Timeout.throw()

    def test_validation_is_value_error(self):
        with self.assertRaises(ValueError):
            AppException.NotAnInteger.throw()

    def test_validation_is_application_error(self):
        with self.assertRaises(ApplicationError):
            AppException.NotAList.throw()

    def test_formatted_legacy_message(self):
        with self.assertRaisesRegex(Timeout, r"^101 - Timeout connecting to resource\.$"):
            AppException.Timeout.throw()

    def test_custom_message(self):
        with self.assertRaisesRegex(Timeout, r"^101 - Custom text$"):
            AppException.Timeout.throw("Custom text")

    def test_override_does_not_mutate_catalog(self):
        AppException.Timeout.create("Custom")
        self.assertEqual(AppException.Timeout.message, "Timeout connecting to resource.")

    def test_error_attributes(self):
        error = AppException.Timeout.create(context={"attempt": 2})
        self.assertEqual((error.code, error.key, error.message),
                         (101, "Timeout", "Timeout connecting to resource."))
        self.assertEqual(error.context["attempt"], 2)

    def test_context_defaults_to_empty(self):
        self.assertEqual(AppException.Generic.create().to_dict()["context"], {})

    def test_context_is_deeply_copied(self):
        original = {"outer": {"numbers": [1, 2]}}
        error = AppException.Timeout.create(context=original)
        original["outer"]["numbers"].append(3)
        self.assertEqual(error.to_dict()["context"], {"outer": {"numbers": [1, 2]}})

    def test_context_top_level_immutable(self):
        error = AppException.Timeout.create(context={"x": 1})
        with self.assertRaises(TypeError):
            error.context["x"] = 2

    def test_context_nested_mapping_immutable(self):
        error = AppException.Timeout.create(context={"a": {"b": 1}})
        with self.assertRaises(TypeError):
            error.context["a"]["b"] = 2

    def test_context_nested_sequence_immutable(self):
        error = AppException.Timeout.create(context={"seq": [1, 2]})
        self.assertIsInstance(error.context["seq"], tuple)

    def test_to_dict_is_detached(self):
        error = AppException.Timeout.create(context={"items": [1]})
        payload = error.to_dict()
        payload["context"]["items"].append(2)
        self.assertEqual(error.to_dict()["context"]["items"], [1])

    def test_json_round_trip(self):
        source = AppException.Timeout.create("Timed out", context={"tags": ["a", 2]})
        restored = AppException.from_json(source.to_json())
        self.assertIsInstance(restored, Timeout)
        self.assertEqual(restored.to_dict(), source.to_dict())

    def test_unicode_round_trip(self):
        source = AppException.Generic.create("Échec", context={"ville": "Sofia"})
        self.assertEqual(AppException.from_json(source.to_json()).to_dict(), source.to_dict())

    def test_json_payload_has_only_four_keys(self):
        error = AppException.Timeout.create()
        self.assertEqual(set(json.loads(error.to_json())), {"code", "key", "message", "context"})

    def test_catalog_json_is_readable(self):
        rows = json.loads(AppException.catalog_json())
        self.assertEqual(rows[1]["key"], "Timeout")
        self.assertEqual(rows[1]["type"], "Timeout")

    def test_catalog_snapshot_is_tuple(self):
        self.assertIsInstance(AppException.catalog(), tuple)

    def test_invalid_error_code(self):
        data = AppException.Timeout.create().to_dict()
        data["code"] = 999
        with self.assertRaises(ValueError):
            AppException.from_dict(data)

    def test_mismatched_key_rejected(self):
        data = AppException.Timeout.create().to_dict()
        data["key"] = "Generic"
        with self.assertRaises(ValueError):
            AppException.from_dict(data)

    def test_bool_code_rejected(self):
        data = AppException.Timeout.create().to_dict()
        data["code"] = True
        with self.assertRaises(TypeError):
            AppException.from_dict(data)

    def test_extra_payload_fields_rejected(self):
        data = AppException.Timeout.create().to_dict()
        data["traceback"] = "secret"
        with self.assertRaises(ValueError):
            AppException.from_dict(data)

    def test_missing_payload_fields_rejected(self):
        with self.assertRaises(ValueError):
            AppException.from_dict({"code": 101, "key": "Timeout"})

    def test_non_mapping_payload_rejected(self):
        with self.assertRaises(TypeError):
            AppException.from_dict([])

    def test_invalid_json_rejected(self):
        with self.assertRaises(json.JSONDecodeError):
            AppException.from_json("{bad")

    def test_nonstandard_json_constant_rejected(self):
        data = AppException.Timeout.create().to_dict()
        data["context"]["x"] = float("nan")
        text = json.dumps(data)
        with self.assertRaises(ValueError):
            AppException.from_json(text)

    def test_bad_context_type(self):
        with self.assertRaises(TypeError):
            AppException.Timeout.create(context=["not-a-map"])

    def test_non_json_context_rejected(self):
        with self.assertRaises(TypeError):
            AppException.Timeout.create(context={"object": object()})

    def test_non_string_context_key_rejected(self):
        with self.assertRaises(TypeError):
            AppException.Timeout.create(context={1: "value"})

    def test_nan_context_rejected(self):
        with self.assertRaises(ValueError):
            AppException.Timeout.create(context={"n": float("nan")})

    def test_blank_message_rejected(self):
        with self.assertRaises(ValueError):
            AppException.Timeout.create(" ")

    def test_bad_message_type_rejected(self):
        with self.assertRaises(TypeError):
            AppException.Timeout.create(123)

    def test_invalid_cause_rejected(self):
        with self.assertRaises(TypeError):
            AppException.Timeout.throw(cause="not an exception")

    def test_chaining_preserves_original_cause(self):
        cause = OSError("network issue")
        with self.assertRaises(Timeout) as captured:
            AppException.Timeout.throw(cause=cause)
        self.assertIs(captured.exception.__cause__, cause)

    def test_no_cause_by_default(self):
        error = AppException.Timeout.create()
        self.assertIsNone(error.__cause__)


class TestConfigurationValidation(unittest.TestCase):
    def test_valid_config_with_default_timeout(self):
        self.assertEqual(validate_configuration({"retries": 2, "hosts": ["a"]}),
                         {"retries": 2, "hosts": ["a"], "timeout_seconds": 30})

    def test_valid_config_with_explicit_timeout(self):
        self.assertEqual(validate_configuration({"retries": 0, "hosts": ["a", "b"],
                                                 "timeout_seconds": 1})["timeout_seconds"], 1)

    def test_valid_config_detaches_hosts(self):
        hosts = ["one"]
        normalized = validate_configuration({"retries": 1, "hosts": hosts})
        hosts.append("two")
        self.assertEqual(normalized["hosts"], ["one"])

    def test_bad_config_type(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration([])
        self.assertEqual(captured.exception.code, 201)

    def test_missing_retries(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"hosts": ["a"]})
        self.assertEqual(captured.exception.code, 202)

    def test_missing_hosts(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1})
        self.assertEqual(captured.exception.code, 202)

    def test_unexpected_field(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1, "hosts": ["a"], "other": 3})
        self.assertEqual(captured.exception.code, 205)

    def test_boolean_retries_are_not_integers(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": True, "hosts": ["a"]})
        self.assertEqual(captured.exception.code, 200)

    def test_retries_lower_boundary(self):
        self.assertEqual(validate_configuration({"retries": 0, "hosts": ["a"]})["retries"], 0)

    def test_retries_upper_boundary(self):
        self.assertEqual(validate_configuration({"retries": 10, "hosts": ["a"]})["retries"], 10)

    def test_retries_outside_bounds(self):
        for bad in (-1, 11):
            with self.subTest(value=bad), self.assertRaises(ValidationException) as capture:
                validate_configuration({"retries": bad, "hosts": ["a"]})
            self.assertEqual(capture.exception.code, 204)

    def test_hosts_not_list(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1, "hosts": ("a",)})
        self.assertEqual(captured.exception.code, 201)

    def test_hosts_empty(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1, "hosts": []})
        self.assertEqual(captured.exception.code, 204)

    def test_host_not_string(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1, "hosts": [5]})
        self.assertEqual(captured.exception.code, 203)

    def test_blank_host(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1, "hosts": ["  "]})
        self.assertEqual(captured.exception.code, 204)

    def test_timeout_not_integer(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1, "hosts": ["a"], "timeout_seconds": False})
        self.assertEqual(captured.exception.code, 200)

    def test_timeout_nonpositive(self):
        with self.assertRaises(ValidationException) as captured:
            validate_configuration({"retries": 1, "hosts": ["a"], "timeout_seconds": 0})
        self.assertEqual(captured.exception.code, 204)

    def test_config_no_mutation(self):
        source = {"retries": 1, "hosts": ["a"]}
        _ = validate_configuration(source)
        self.assertEqual(source, {"retries": 1, "hosts": ["a"]})


suite = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(TestExtendedExceptionCatalog),
    unittest.defaultTestLoader.loadTestsFromTestCase(TestConfigurationValidation),
])
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful(), "Extended notebook tests failed"
print(f"ALL TESTS PASSED (extended: {result.testsRun})")

..........................................................
----------------------------------------------------------------------
Ran 58 tests in 0.057s

OK


ALL TESTS PASSED (extended: 58)


## 6. Operational recommendations and limitations

- Treat **error codes as stable API identifiers**, ideally with an organization-wide allocation policy. Enum uniqueness only enforces uniqueness within this catalog.
- Keep application exceptions distinct from process-control exceptions (`KeyboardInterrupt`, `SystemExit`). `cause=` enables debugging without exposing original traceback in serialized output.
- Log error codes and vetted context; do not log credentials, authentication tokens, or raw user content. This example does not automatically redact arbitrary text.
- This serializer reconstructs only **known catalog errors**; it is not a generic exception-deserialization mechanism. Do not treat a reconstructed exception as authorization evidence.
- A direct `ApplicationError(...)` call remains possible for advanced use, but normal application code should go through `AppException` for catalog consistency.
- The configuration validator is an illustration, not a complete schema language. For a larger application consider dedicated schema validation, localization, versioning, observability integration and documented migration of retired codes.